# NB16 — Locked Paired Significance Summary (Herbal Supplements)

**Purpose.** The LOCKED confirmatory inference notebook. Strict `case_id` pairing on the fixed primary outcome (NDCG@5 at report pool depth 1000) drawn from the NB14 canonical rows; regime-stratified paired bootstrap (base seed 42; sign-flip 10,000 iterations, seed 42), McNemar, and Holm adjustment over the pre-registered confirmatory family `principal_stage2_overall_8` (α = 0.05, m = 8 = {Shannon, GAM, LightGBM, Transformer} × {P2-P − P2-Q, Full − P2-P}); Stage-1 exact McNemar on HitRate@1000 at depth 1000. Coverage diagnostics are exported BEFORE the hard coverage gate, and SELF-CHECK SC-3 validates the locked Holm family (membership, rank order, adjusted-p monotonicity, negative controls) before export.
**Inputs.** NB14 `pipeline_manifest.json` (validated before outcome loading) and the canonical primary parquet; expected regimes [cold, weak, strong].
**Outputs.** The paired-significance export suite: coverage diagnostics, per-contrast bootstrap/sign-flip/McNemar results, the Holm-adjusted confirmatory table, and the run manifest.
**Position.** NB14 → **this** → acceptance verdicts consumed by Chapters 6–8 and by the acceptance probes. The sealed raw p-values and accepted deltas of record (G-4) live in this notebook's exported artefacts.
**Run notes.** Executed artefact of record — do not re-run; the exported results feed the downstream chain-gate. Bootstrap and sign-flip seeds are locked (G-3); the inference definitions cell below is part of the locked contract.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ==== Imports ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

from scipy.stats import binomtest


In [4]:
# ==== Locked Analysis Contract ====
NOTEBOOK_NAME = "16_paired_significance_summary_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
REVISION_DATE = "2026-07-21"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PIPELINE_AGGREGATE_DIR = OUTPUTS_DIR / "pipeline_aggregate"
CANONICAL_METRICS_PATH = (
    PIPELINE_AGGREGATE_DIR / "pipeline_canonical_per_case_metrics.parquet"
)
CANONICAL_PRIMARY_PATH = (
    PIPELINE_AGGREGATE_DIR
    / "pipeline_canonical_primary_ndcg5_depth1000.parquet"
)
PIPELINE_MANIFEST_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_manifest.json"

OUT_DIR = OUTPUTS_DIR / "analysis" / "paired_significance_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "paired_case_deltas_report_depth": OUT_DIR / "paired_case_deltas_report_depth.parquet",
    "paired_inference_summary_report_depth": OUT_DIR / "paired_inference_summary_report_depth.csv",
    "paired_inference_by_regime_report_depth": OUT_DIR / "paired_inference_by_regime_report_depth.csv",
    "pair_coverage_qc": OUT_DIR / "pair_coverage_qc.csv",
    "pair_missing_case_ids": OUT_DIR / "pair_missing_case_ids.csv",
    "multiplicity_adjustment_summary": OUT_DIR / "multiplicity_adjustment_summary.csv",
    "stage1_mcnemar_hitrate1000": OUT_DIR / "stage1_mcnemar_hitrate1000.csv",
    "run_manifest": OUT_DIR / "run_manifest.json",
}

PRIMARY_METRIC_NAME = "NDCG"
PRIMARY_METRIC_CUTOFF = 5
REPORT_POOL_DEPTH = 1000
PRIMARY_METHOD_FAMILIES = ["shannon", "gam", "lightgbm", "transformer"]
METHOD_FAMILY_LABELS = {'shannon': 'Shannon', 'gam': 'GAM', 'lightgbm': 'LightGBM', 'transformer': 'Transformer'}
EXPECTED_REGIMES = ['cold', 'weak', 'strong']
SCOPES = ["overall", *EXPECTED_REGIMES, "non-cold"]
SCOPE_ORDER = {scope: index for index, scope in enumerate(SCOPES)}

BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_BASE_SEED = 42
SIGN_FLIP_ITERATIONS = 10_000
SIGN_FLIP_SEED = 42
SIMULATION_BATCH_SIZE = 256
DELTA_TIE_ATOL = 1e-12
HOLM_ALPHA = 0.05
CONFIRMATORY_HOLM_FAMILY_ID = "principal_stage2_overall_8"
PRINCIPAL_CONTRAST_CODES = {
    "P2_P_minus_P2_Q",
    "Full_minus_P2_P",
}
EXPECTED_CONFIRMATORY_FAMILY = {
    (family, contrast_code)
    for family in PRIMARY_METHOD_FAMILIES
    for contrast_code in PRINCIPAL_CONTRAST_CODES
}

STAGE1_METRIC_NAME = "HitRate"
STAGE1_METRIC_CUTOFF = 1000
STAGE1_POOL_DEPTH = 1000

CANONICAL_READ_COLUMNS = [
    "case_id", "query_id", "user_id", "regime", "stage_condition",
    "reranker_method", "method_family", "candidate_pool_depth",
    "metric_name", "metric_cutoff", "metric_value", "target_parent_asin",
    "target_exposed", "candidate_source", "record_type",
    "shared_stage1_baseline", "shared_baseline_repeated_for_display",
]
AUTHORITATIVE_ROW_FILTERS = [
    ("metric_name", "==", PRIMARY_METRIC_NAME),
    ("metric_cutoff", "==", PRIMARY_METRIC_CUTOFF),
    ("candidate_pool_depth", "==", REPORT_POOL_DEPTH),
]

COMPARISON_SPECS = [{
    "comparison_order": 1,
    "comparison_id": "P1_only_minus_P0",
    "contrast_code": "P1_only_minus_P0",
    "comparison_label": "P1-only − P0",
    "comparison_method_family": "shared_retrieval_baseline",
    "method_family_label": "Shared retrieval",
    "left_condition": "P1-only",
    "right_condition": "P0",
    "left_source_method_family": "shared_retrieval_baseline",
    "right_source_method_family": "shared_retrieval_baseline",
    "method_match_rule": "shared_retrieval_only",
    "inference_role": "stage1_separate_mcnemar",
    "multiplicity_family_id": "stage1_unadjusted",
}]

_comparison_order = 2
for _family in PRIMARY_METHOD_FAMILIES:
    _label = METHOD_FAMILY_LABELS[_family]
    for _spec in [
        {
            "contrast_code": "P2_Q_minus_P0",
            "comparison_label": "P2-Q − P0",
            "left_condition": "P2-Q",
            "right_condition": "P0",
            "right_source_method_family": "shared_retrieval_baseline",
            "inference_role": "outside_holm_unadjusted",
        },
        {
            "contrast_code": "P2_P_minus_P2_Q",
            "comparison_label": "P2-P − P2-Q",
            "left_condition": "P2-P",
            "right_condition": "P2-Q",
            "right_source_method_family": _family,
            "inference_role": "principal_confirmatory",
        },
        {
            "contrast_code": "Full_minus_P2_P",
            "comparison_label": "Full − P2-P",
            "left_condition": "Full",
            "right_condition": "P2-P",
            "right_source_method_family": _family,
            "inference_role": "principal_confirmatory",
        },
        {
            "contrast_code": "Full_minus_P2_Q",
            "comparison_label": "Full − P2-Q",
            "left_condition": "Full",
            "right_condition": "P2-Q",
            "right_source_method_family": _family,
            "inference_role": "exploratory_unadjusted",
        },
    ]:
        COMPARISON_SPECS.append({
            "comparison_order": _comparison_order,
            "comparison_id": f"{_family}_{_spec['contrast_code']}",
            "contrast_code": _spec["contrast_code"],
            "comparison_label": _spec["comparison_label"],
            "comparison_method_family": _family,
            "method_family_label": _label,
            "left_condition": _spec["left_condition"],
            "right_condition": _spec["right_condition"],
            "left_source_method_family": _family,
            "right_source_method_family": _spec["right_source_method_family"],
            "method_match_rule": (
                "same_method_family"
                if _spec["right_source_method_family"] == _family
                else f"fixed_{_family}_vs_shared_retrieval_baseline"
            ),
            "inference_role": _spec["inference_role"],
            "multiplicity_family_id": _spec["inference_role"],
        })
        _comparison_order += 1

if len(COMPARISON_SPECS) != 17:
    raise RuntimeError("The locked comparison registry must contain 17 comparisons.")
if len({spec["comparison_id"] for spec in COMPARISON_SPECS}) != len(COMPARISON_SPECS):
    raise RuntimeError("The locked comparison registry contains duplicate IDs.")

stage2_conditions = {"P2-Q", "P2-P", "Full"}
for comparison_spec in COMPARISON_SPECS:
    if (
        comparison_spec["left_condition"] in stage2_conditions
        and comparison_spec["right_condition"] in stage2_conditions
        and comparison_spec["left_source_method_family"] != comparison_spec["right_source_method_family"]
    ):
        raise RuntimeError("A locked Stage-2 comparison is not method-family matched.")
    allowed_families = ["shared_retrieval_baseline", *PRIMARY_METHOD_FAMILIES]
    if comparison_spec["comparison_method_family"] not in allowed_families:
        raise RuntimeError("Unexpected comparison method family.")

print("Category:", CATEGORY_LABEL)
print("Locked endpoint: unconditional NDCG@5 at candidate depth 1000")
print("Locked Holm family:", sorted(EXPECTED_CONFIRMATORY_FAMILY))


Category: Herbal Supplements
Locked endpoint: unconditional NDCG@5 at candidate depth 1000
Locked Holm family: [('gam', 'Full_minus_P2_P'), ('gam', 'P2_P_minus_P2_Q'), ('lightgbm', 'Full_minus_P2_P'), ('lightgbm', 'P2_P_minus_P2_Q'), ('shannon', 'Full_minus_P2_P'), ('shannon', 'P2_P_minus_P2_Q'), ('transformer', 'Full_minus_P2_P'), ('transformer', 'P2_P_minus_P2_Q')]


## Locked Inference Definitions

For each case \(i\), the paired difference is \(d_i=y_{i,L}-y_{i,R}\). The mean paired difference is reported with a 95% regime-stratified paired bootstrap interval. Because the benchmark retains at most one case per user, the paired case and user units are identical.

The two-sided randomization test uses 10,000 paired sign reversals with seed 42. Sign reversals are performed within history-regime strata and recombined at the empirical weights of the reported population.

Within each category, Holm correction is applied only to the eight overall tests in:

\[
\{\mathrm{P2\!-\!P-P2\!-\!Q},\ \mathrm{Full-P2\!-\!P}\}
\times
\{\mathrm{Shannon,GAM,LightGBM,Transformer}\}.
\]

P2-Q minus P0 is outside this family. Full minus P2-Q is exploratory and unadjusted. Regime-specific and non-cold rows are also unadjusted.


In [5]:
# ==== Shared Helpers ====
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def require_columns(frame, required_columns, label):
    missing = sorted(set(required_columns).difference(frame.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(frame, label):
    duplicated = frame.columns[frame.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated columns: {duplicated}")


def boolean_series(series, label):
    if pd.api.types.is_bool_dtype(series):
        parsed = series.astype("boolean")
    else:
        text = series.astype("string").str.strip().str.lower()
        numeric = pd.to_numeric(series, errors="coerce")
        mapped = text.map({
            "true": True,
            "false": False,
            "yes": True,
            "no": False,
            "1": True,
            "0": False,
        })
        parsed = mapped.where(
            mapped.notna(), numeric.map({1.0: True, 0.0: False})
        ).astype("boolean")
    if parsed.isna().any():
        raise RuntimeError(f"{label} contains non-boolean or missing values.")
    return parsed.astype(bool)



def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def make_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_jsonable(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if pd.isna(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return None
    return value


def write_json(path, payload):
    Path(path).write_text(
        json.dumps(make_jsonable(payload), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def scope_mask(frame, scope):
    regime = frame["regime"].astype(str)
    if scope == "overall":
        return pd.Series(True, index=frame.index, dtype=bool)
    if scope == "non-cold":
        return regime.ne("cold")
    if scope in set(EXPECTED_REGIMES):
        return regime.eq(scope)
    raise ValueError(f"Unsupported scope: {scope}")


def side_frame(source, condition, source_method_family, side_label):
    selected = source.loc[
        source["stage_condition"].eq(condition)
        & source["method_family"].eq(source_method_family)
    ].copy()
    if selected.empty:
        raise RuntimeError(
            f"No canonical rows for condition={condition}, method_family={source_method_family}."
        )
    if selected["case_id"].duplicated().any():
        raise RuntimeError(
            f"Duplicate case rows for condition={condition}, method_family={source_method_family}."
        )
    rename_map = {
        column: f"{column}_{side_label}"
        for column in [
            "query_id",
            "user_id",
            "regime",
            "target_parent_asin",
            "reranker_method",
            "method_family",
            "candidate_source",
            "target_exposed",
            "metric_value",
        ]
    }
    return selected[["case_id", *rename_map]].rename(columns=rename_map)


In [6]:
# ==== Validate the Notebook 14 Manifest Before Outcome Loading ====
required_paths = [CANONICAL_METRICS_PATH, CANONICAL_PRIMARY_PATH, PIPELINE_MANIFEST_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing authoritative Notebook 14 artifacts: {missing_paths}")

pipeline_manifest = load_json(PIPELINE_MANIFEST_PATH)
if pipeline_manifest.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 14 manifest category_id does not match this notebook.")
if pipeline_manifest.get("run_status") != "SUCCESS":
    raise RuntimeError("Notebook 14 is not sealed with run_status=SUCCESS.")
if pipeline_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 14 is not ready for downstream inference.")
if pipeline_manifest.get("full_transfer_readiness") is not True:
    raise RuntimeError("Notebook 14 Full transfer readiness failed.")
if pipeline_manifest.get("canonical_raw_sha256") != file_sha256(
    CANONICAL_METRICS_PATH
):
    raise RuntimeError("Notebook 14 canonical raw SHA mismatch.")
if pipeline_manifest.get("canonical_primary_sha256") != file_sha256(
    CANONICAL_PRIMARY_PATH
):
    raise RuntimeError("Notebook 14 canonical primary-grid SHA mismatch.")

manifest_output_paths = pipeline_manifest.get("output_paths")
if not isinstance(manifest_output_paths, dict):
    raise RuntimeError("Notebook 14 manifest is missing output_paths.")
if Path(manifest_output_paths.get("canonical_raw", "")) != CANONICAL_METRICS_PATH:
    raise RuntimeError("Notebook 14 canonical_raw path does not match the configured input.")
if Path(manifest_output_paths.get("canonical_primary", "")) != CANONICAL_PRIMARY_PATH:
    raise RuntimeError("Notebook 14 canonical_primary path does not match the configured input.")
if Path(manifest_output_paths.get("manifest", "")) != PIPELINE_MANIFEST_PATH:
    raise RuntimeError("Notebook 14 manifest path does not match the configured input.")

validation_results_14 = pipeline_manifest.get("validation_results")
if not isinstance(validation_results_14, dict):
    raise RuntimeError("Notebook 14 manifest is missing validation_results.")
required_notebook14_checks = [
    "all_five_conditions",
    "all_reranker_families",
    "shared_baselines_stored_once",
    "pool_depth_and_metric_cutoff_separate",
    "candidate_counts_within_pool",
    "target_rank_one_based",
    "target_rank_within_candidate_count",
    "target_absent_metrics_zero",
    "no_duplicated_canonical_key",
    "no_model_selection",
    "all_source_metrics_validated",
    "paired_case_universes_match",
    "qchs_fallback_cases_preserved",
    "no_missing_expected_cells",
    "upstream_readiness_passed",
    "full_transfer_readiness",
    "primary_ndcg5_depth1000_grid_complete",
    "canonical_source_hashes_complete",
]
failed_notebook14_checks = [
    key for key in required_notebook14_checks
    if validation_results_14.get(key) is not True
]
if failed_notebook14_checks:
    raise RuntimeError(
        f"Notebook 14 manifest did not pass required canonical-rank/pairing QC: {failed_notebook14_checks}"
    )

pool_contract = pipeline_manifest.get("pool_depth_contract", {})
if REPORT_POOL_DEPTH not in [int(value) for value in pool_contract.get("candidate_pool_depths", [])]:
    raise RuntimeError("Notebook 14 manifest does not include the fixed report pool depth.")
if PRIMARY_METRIC_CUTOFF not in [int(value) for value in pool_contract.get("metric_cutoffs", [])]:
    raise RuntimeError("Notebook 14 manifest does not include the fixed primary cutoff.")
if pool_contract.get("dimensions_are_distinct") is not True:
    raise RuntimeError("Notebook 14 did not keep pool depth and metric cutoff distinct.")

stage1_winner_lineage = pipeline_manifest.get("stage1_winner_lineage", {})
if stage1_winner_lineage.get("lineage_validated") is not True:
    raise RuntimeError("Notebook 14 stage winner lineage is not validated.")

print("Notebook 14 manifest validation: PASS")
print("Category-specific winner lineage:", stage1_winner_lineage)


Notebook 14 manifest validation: PASS
Category-specific winner lineage: {'notebook07_winner_manifest_path': '/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_query_retrieval_selection/stage1_query_only_winner_herbal.json', 'notebook08_winner_manifest_path': '/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_winner_herbal.json', 'query_only_method_key': 'hybrid_dense_bm25', 'personalized_method_slug': 'profile_sparse_qcha', 'lineage_validated': True}


In [7]:
# ==== Load Only Fixed NDCG@5 / Depth-1000 Canonical Outcome Rows ====
canonical_metrics = pd.read_parquet(
    CANONICAL_METRICS_PATH,
    columns=CANONICAL_READ_COLUMNS,
    filters=AUTHORITATIVE_ROW_FILTERS,
).copy()
canonical_primary_grid = pd.read_parquet(
    CANONICAL_PRIMARY_PATH
).copy()
primary_grid_required = [
    "case_id", "stage_condition", "analysis_method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff", "metric_value",
]
require_columns(
    canonical_primary_grid, primary_grid_required,
    "Notebook 14 canonical primary method grid",
)
require_unique_columns(
    canonical_primary_grid, "Notebook 14 canonical primary method grid"
)
_primary_grid_key = [
    "case_id", "stage_condition", "analysis_method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
if canonical_primary_grid.duplicated(_primary_grid_key).any():
    raise RuntimeError("Notebook 14 canonical primary method grid has duplicate keys.")
# Notebook 14 now also carries the b02 (RetrP) arm of the 2x2 stage
# allocation. This notebook's locked comparison registry is confined to the
# five conditions below; b02 is therefore required to be present upstream but
# is not admitted into the inferential slice built here.
INFERENTIAL_CONDITIONS = {"P0", "P1-only", "P2-Q", "P2-P", "Full"}
UPSTREAM_ONLY_CONDITIONS = {"b02"}
_grid_conditions = set(canonical_primary_grid["stage_condition"].astype(str))
if not INFERENTIAL_CONDITIONS.issubset(_grid_conditions):
    raise RuntimeError(
        "Notebook 14 canonical primary grid is missing a condition: "
        f"missing={sorted(INFERENTIAL_CONDITIONS - _grid_conditions)}"
    )
if not _grid_conditions.issubset(INFERENTIAL_CONDITIONS | UPSTREAM_ONLY_CONDITIONS):
    raise RuntimeError(
        "Notebook 14 canonical primary grid carries an unrecognised condition: "
        f"extra={sorted(_grid_conditions - INFERENTIAL_CONDITIONS - UPSTREAM_ONLY_CONDITIONS)}"
    )
if set(canonical_primary_grid["analysis_method_family"].astype(str)) != set(
    PRIMARY_METHOD_FAMILIES
):
    raise RuntimeError("Notebook 14 canonical primary grid is missing a method family.")
_primary_grid_case_count = canonical_primary_grid["case_id"].astype(str).nunique()
# The upstream grid is a design registry: (shannon, b02) is undefined because
# the training-free Shannon reranker has no fitting step, so the grid is not
# |conditions| x |families|. Completeness is verified per observed design cell.
_observed_grid_pairs = set(
    canonical_primary_grid[["analysis_method_family", "stage_condition"]]
    .astype(str).drop_duplicates().itertuples(index=False, name=None)
)
_expected_grid_pairs = {
    (_family, _condition)
    for _family in PRIMARY_METHOD_FAMILIES
    for _condition in sorted(INFERENTIAL_CONDITIONS)
}
if not _expected_grid_pairs.issubset(_observed_grid_pairs):
    raise RuntimeError(
        "Notebook 14 canonical primary grid is missing an inferential cell: "
        f"missing={sorted(_expected_grid_pairs - _observed_grid_pairs)}"
    )
_expected_primary_grid_rows = _primary_grid_case_count * len(_observed_grid_pairs)
if len(canonical_primary_grid) != _expected_primary_grid_rows:
    raise RuntimeError(
        "Notebook 14 canonical primary grid is incomplete: "
        f"expected={_expected_primary_grid_rows}, "
        f"observed={len(canonical_primary_grid)}"
    )
# Restrict the grid used by this notebook to the locked five-condition slice.
canonical_primary_grid = canonical_primary_grid.loc[
    canonical_primary_grid["stage_condition"].astype(str).isin(INFERENTIAL_CONDITIONS)
].reset_index(drop=True)
if len(canonical_primary_grid) != (
    _primary_grid_case_count * len(_expected_grid_pairs)
):
    raise RuntimeError(
        "The five-condition inferential slice of the Notebook 14 primary grid "
        "is incomplete after restriction."
    )

require_columns(canonical_metrics, CANONICAL_READ_COLUMNS, "Notebook 14 canonical metrics")
require_unique_columns(canonical_metrics, "Notebook 14 canonical metrics")
if canonical_metrics.empty:
    raise RuntimeError("The fixed NDCG@5, depth-1000 Notebook 14 slice is empty.")

for column in [
    "case_id",
    "query_id",
    "user_id",
    "regime",
    "stage_condition",
    "reranker_method",
    "method_family",
    "target_parent_asin",
    "candidate_source",
    "record_type",
]:
    canonical_metrics[column] = (
        canonical_metrics[column].fillna("").astype(str).map(normalize_space)
    )
canonical_metrics["regime"] = canonical_metrics["regime"].str.lower()
canonical_metrics["method_family"] = canonical_metrics["method_family"].str.lower()
canonical_metrics["candidate_pool_depth"] = pd.to_numeric(
    canonical_metrics["candidate_pool_depth"], errors="raise"
).astype(int)
canonical_metrics["metric_cutoff"] = pd.to_numeric(
    canonical_metrics["metric_cutoff"], errors="raise"
).astype(int)
canonical_metrics["metric_value"] = pd.to_numeric(
    canonical_metrics["metric_value"], errors="raise"
).astype(float)
canonical_metrics["target_exposed"] = boolean_series(
    canonical_metrics["target_exposed"], "target_exposed"
)
canonical_metrics["shared_stage1_baseline"] = boolean_series(
    canonical_metrics["shared_stage1_baseline"], "shared_stage1_baseline"
)
canonical_metrics["shared_baseline_repeated_for_display"] = boolean_series(
    canonical_metrics["shared_baseline_repeated_for_display"],
    "shared_baseline_repeated_for_display",
)

if not canonical_metrics["metric_name"].eq(PRIMARY_METRIC_NAME).all():
    raise RuntimeError("Outcome rows outside the fixed NDCG metric were loaded.")
if not canonical_metrics["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF).all():
    raise RuntimeError("Outcome rows outside the fixed cutoff 5 were loaded.")
if not canonical_metrics["candidate_pool_depth"].eq(REPORT_POOL_DEPTH).all():
    raise RuntimeError("Outcome rows outside the fixed pool depth 1000 were loaded.")
if canonical_metrics["metric_value"].isna().any():
    raise RuntimeError("Canonical NDCG@5 contains missing values.")
if not canonical_metrics["metric_value"].between(0.0, 1.0, inclusive="both").all():
    raise RuntimeError("Canonical NDCG@5 falls outside [0, 1].")
if canonical_metrics[["case_id", "query_id", "user_id", "regime"]].eq("").any().any():
    raise RuntimeError("Canonical outcome rows contain empty case/query/user/regime identifiers.")
if canonical_metrics["shared_baseline_repeated_for_display"].any():
    raise RuntimeError("Display-only repeated baseline rows entered inferential data.")

absent_values = canonical_metrics.loc[
    ~canonical_metrics["target_exposed"], "metric_value"
].to_numpy(dtype=float)
if not np.isclose(absent_values, 0.0, rtol=0.0, atol=0.0).all():
    raise RuntimeError("Target-absent cases were not preserved as NDCG@5 = 0.")

expected_conditions = set(INFERENTIAL_CONDITIONS)
_canonical_conditions_observed = set(canonical_metrics["stage_condition"])
if not expected_conditions.issubset(_canonical_conditions_observed):
    raise RuntimeError(
        "The fixed canonical slice does not contain all five pipeline conditions: "
        f"missing={sorted(expected_conditions - _canonical_conditions_observed)}"
    )
if not _canonical_conditions_observed.issubset(
    expected_conditions | UPSTREAM_ONLY_CONDITIONS
):
    raise RuntimeError(
        "The canonical slice carries an unrecognised condition: "
        f"extra={sorted(_canonical_conditions_observed - expected_conditions - UPSTREAM_ONLY_CONDITIONS)}"
    )
# b02 (RetrP) is materialised upstream but lies outside this notebook's locked
# comparison registry; it is excluded here so that the case-identity, pairing,
# and grain invariants below are evaluated on the inferential slice only.
_b02_row_count = int(
    canonical_metrics["stage_condition"].isin(UPSTREAM_ONLY_CONDITIONS).sum()
)
canonical_metrics = canonical_metrics.loc[
    canonical_metrics["stage_condition"].isin(expected_conditions)
].reset_index(drop=True)
print(
    f"Excluded {_b02_row_count} upstream-only b02 rows "
    "(outside the locked comparison registry of this notebook)."
)

identity_columns = ["query_id", "user_id", "regime", "target_parent_asin"]
identity_nunique = canonical_metrics.groupby("case_id")[identity_columns].nunique(dropna=False)
if identity_nunique.gt(1).any().any():
    raise RuntimeError("Case identity or regime is inconsistent across canonical conditions.")

case_user_pairs = canonical_metrics[["case_id", "user_id"]].drop_duplicates()
if case_user_pairs["case_id"].duplicated().any():
    raise RuntimeError("A case_id maps to more than one user_id.")
if case_user_pairs["user_id"].duplicated().any():
    raise RuntimeError("The one-case-per-user design is violated.")

observed_regimes = set(canonical_metrics["regime"])
if observed_regimes != set(EXPECTED_REGIMES):
    raise RuntimeError(
        f"Unexpected category regime set: observed={sorted(observed_regimes)}, "
        f"expected={EXPECTED_REGIMES}"
    )

shared_rows = canonical_metrics.loc[
    canonical_metrics["stage_condition"].isin(["P0", "P1-only"])
].copy()
if (
    not shared_rows["method_family"].eq("shared_retrieval_baseline").all()
    or not shared_rows["reranker_method"].eq("none").all()
    or not shared_rows["shared_stage1_baseline"].all()
):
    raise RuntimeError("P0/P1-only are not stored once as the shared retrieval baseline.")

stage2_rows = canonical_metrics.loc[
    canonical_metrics["stage_condition"].isin(["P2-Q", "P2-P", "Full"])
    & canonical_metrics["method_family"].isin(PRIMARY_METHOD_FAMILIES)
].copy()
primary_metrics = pd.concat([shared_rows, stage2_rows], ignore_index=True, sort=False)
primary_key = ["case_id", "stage_condition", "method_family"]
if primary_metrics.duplicated(primary_key).any():
    raise RuntimeError("More than one row exists per case_id × condition × method_family.")

for method_family in PRIMARY_METHOD_FAMILIES:
    for condition in ["P2-Q", "P2-P", "Full"]:
        cell = primary_metrics.loc[
            primary_metrics["method_family"].eq(method_family)
            & primary_metrics["stage_condition"].eq(condition)
        ]
        if cell.empty:
            raise RuntimeError(f"Missing primary cell: {method_family} / {condition}")
        if cell["reranker_method"].nunique() != 1:
            raise RuntimeError(
                f"Multiple reranker methods occur within fixed family/stage: {method_family} / {condition}"
            )

method_lineage = (
    primary_metrics[[
        "stage_condition", "method_family", "reranker_method", "candidate_source"
    ]]
    .drop_duplicates()
    .sort_values(["method_family", "stage_condition"], kind="mergesort")
    .to_dict("records")
)

print("Fixed canonical outcome rows:", len(primary_metrics))
print("Unique cases:", primary_metrics["case_id"].nunique())
print("Validation: fixed metric/depth, target-absence, identity, and grain passed")


Excluded 5904 upstream-only b02 rows (outside the locked comparison registry of this notebook).
Fixed canonical outcome rows: 27552
Unique cases: 1968
Validation: fixed metric/depth, target-absence, identity, and grain passed


In [8]:
# ==== Strict case_id Pairing and Coverage QC ====
pair_delta_frames = []
coverage_rows = []
missing_case_rows = []
identity_fields = ["query_id", "user_id", "regime", "target_parent_asin"]

for comparison_spec in COMPARISON_SPECS:
    left = side_frame(
        primary_metrics,
        comparison_spec["left_condition"],
        comparison_spec["left_source_method_family"],
        "left",
    )
    right = side_frame(
        primary_metrics,
        comparison_spec["right_condition"],
        comparison_spec["right_source_method_family"],
        "right",
    )
    merged = left.merge(
        right,
        on="case_id",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    paired = merged.loc[merged["_merge"].eq("both")].copy()

    mismatch_counts = {
        field: int(
            (~paired[f"{field}_left"].eq(paired[f"{field}_right"])).sum()
        )
        for field in identity_fields
    }

    for scope in SCOPES:
        left_scope = left.loc[scope_mask(
            left.rename(columns={"regime_left": "regime"}), scope
        )]
        right_scope = right.loc[scope_mask(
            right.rename(columns={"regime_right": "regime"}), scope
        )]
        left_case_ids = set(left_scope["case_id"])
        right_case_ids = set(right_scope["case_id"])
        missing_left_ids = sorted(right_case_ids.difference(left_case_ids))
        missing_right_ids = sorted(left_case_ids.difference(right_case_ids))
        paired_ids = left_case_ids & right_case_ids

        paired_scope = paired.loc[
            paired["case_id"].isin(paired_ids)
        ]
        scope_mismatch_counts = {
            field: int(
                (~paired_scope[f"{field}_left"].eq(paired_scope[f"{field}_right"])).sum()
            )
            for field in identity_fields
        }
        coverage_status = "PASS" if (
            not missing_left_ids
            and not missing_right_ids
            and all(value == 0 for value in scope_mismatch_counts.values())
            and len(paired_ids) > 0
        ) else "FAIL"
        coverage_rows.append({
            "category_id": CATEGORY_ID,
            "comparison_order": int(comparison_spec["comparison_order"]),
            "comparison_id": comparison_spec["comparison_id"],
            "comparison_label": comparison_spec["comparison_label"],
            "multiplicity_family_id": comparison_spec["multiplicity_family_id"],
            "comparison_method_family": comparison_spec["comparison_method_family"],
            "left_condition": comparison_spec["left_condition"],
            "right_condition": comparison_spec["right_condition"],
            "scope": scope,
            "scope_order": int(SCOPE_ORDER[scope]),
            "n_left": int(len(left_case_ids)),
            "n_right": int(len(right_case_ids)),
            "n_pairs": int(len(paired_ids)),
            "missing_left": int(len(missing_left_ids)),
            "missing_right": int(len(missing_right_ids)),
            "query_id_mismatch_count": scope_mismatch_counts["query_id"],
            "user_id_mismatch_count": scope_mismatch_counts["user_id"],
            "regime_mismatch_count": scope_mismatch_counts["regime"],
            "target_item_mismatch_count": scope_mismatch_counts["target_parent_asin"],
            "coverage_status": coverage_status,
        })

        for case_id in missing_left_ids:
            missing_case_rows.append({
                "category_id": CATEGORY_ID,
                "comparison_id": comparison_spec["comparison_id"],
                "scope": scope,
                "case_id": case_id,
                "missing_side": "left",
                "missing_condition": comparison_spec["left_condition"],
                "available_condition": comparison_spec["right_condition"],
            })
        for case_id in missing_right_ids:
            missing_case_rows.append({
                "category_id": CATEGORY_ID,
                "comparison_id": comparison_spec["comparison_id"],
                "scope": scope,
                "case_id": case_id,
                "missing_side": "right",
                "missing_condition": comparison_spec["right_condition"],
                "available_condition": comparison_spec["left_condition"],
            })

    if any(value != 0 for value in mismatch_counts.values()):
        continue

    pair_delta = pd.DataFrame({
        "category_id": CATEGORY_ID,
        "category_label": CATEGORY_LABEL,
        "comparison_order": int(comparison_spec["comparison_order"]),
        "comparison_id": comparison_spec["comparison_id"],
        "comparison_label": comparison_spec["comparison_label"],
        "contrast_code": comparison_spec["contrast_code"],
        "inference_role": comparison_spec["inference_role"],
        "multiplicity_family_id": comparison_spec["multiplicity_family_id"],
        "comparison_method_family": comparison_spec["comparison_method_family"],
        "method_family_label": comparison_spec["method_family_label"],
        "left_condition": comparison_spec["left_condition"],
        "right_condition": comparison_spec["right_condition"],
        "method_match_rule": comparison_spec["method_match_rule"],
        "method_family_match_pass": True,
        "case_id": paired["case_id"].astype(str),
        "query_id": paired["query_id_left"].astype(str),
        "user_id": paired["user_id_left"].astype(str),
        "regime": paired["regime_left"].astype(str),
        "is_non_cold": paired["regime_left"].astype(str).ne("cold"),
        "target_parent_asin": paired["target_parent_asin_left"].astype(str),
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "left_source_method_family": paired["method_family_left"].astype(str),
        "right_source_method_family": paired["method_family_right"].astype(str),
        "left_reranker_method": paired["reranker_method_left"].astype(str),
        "right_reranker_method": paired["reranker_method_right"].astype(str),
        "left_candidate_source": paired["candidate_source_left"].astype(str),
        "right_candidate_source": paired["candidate_source_right"].astype(str),
        "left_target_exposed": paired["target_exposed_left"].astype(bool),
        "right_target_exposed": paired["target_exposed_right"].astype(bool),
        "left_metric_value": paired["metric_value_left"].astype(float),
        "right_metric_value": paired["metric_value_right"].astype(float),
    })
    pair_delta["delta"] = (
        pair_delta["left_metric_value"] - pair_delta["right_metric_value"]
    )
    pair_delta["delta_direction"] = np.select(
        [
            pair_delta["delta"].gt(DELTA_TIE_ATOL),
            pair_delta["delta"].lt(-DELTA_TIE_ATOL),
        ],
        ["positive", "negative"],
        default="tie",
    )
    pair_delta_frames.append(pair_delta)

paired_case_deltas_report_depth = pd.concat(
    pair_delta_frames, ignore_index=True, sort=False
)
pair_coverage_qc = pd.DataFrame(coverage_rows)
pair_missing_case_ids = pd.DataFrame(
    missing_case_rows,
    columns=[
        "category_id",
        "comparison_id",
        "scope",
        "case_id",
        "missing_side",
        "missing_condition",
        "available_condition",
    ],
)

PAIR_DELTA_COLUMNS = [
    "category_id",
    "category_label",
    "comparison_order",
    "comparison_id",
    "comparison_label",
    "contrast_code",
    "inference_role",
    "multiplicity_family_id",
    "comparison_method_family",
    "method_family_label",
    "left_condition",
    "right_condition",
    "method_match_rule",
    "method_family_match_pass",
    "case_id",
    "query_id",
    "user_id",
    "regime",
    "is_non_cold",
    "target_parent_asin",
    "metric_name",
    "metric_cutoff",
    "candidate_pool_depth",
    "left_source_method_family",
    "right_source_method_family",
    "left_reranker_method",
    "right_reranker_method",
    "left_candidate_source",
    "right_candidate_source",
    "left_target_exposed",
    "right_target_exposed",
    "left_metric_value",
    "right_metric_value",
    "delta",
    "delta_direction",
]
paired_case_deltas_report_depth = paired_case_deltas_report_depth[PAIR_DELTA_COLUMNS]
require_unique_columns(paired_case_deltas_report_depth, "paired case deltas")
pair_key = ["case_id", "comparison_id"]
if paired_case_deltas_report_depth.duplicated(pair_key).any():
    raise RuntimeError("Paired case delta rows are not unique by case_id × comparison_id.")

print("Paired case-delta rows:", len(paired_case_deltas_report_depth))
print("Coverage QC rows:", len(pair_coverage_qc))


Paired case-delta rows: 33456
Coverage QC rows: 85


In [9]:
# ==== Export Coverage Diagnostics Before the Hard Coverage Gate ====
pair_coverage_qc.to_csv(
    OUTPUT_FILES["pair_coverage_qc"], index=False, encoding="utf-8-sig"
)
pair_missing_case_ids.to_csv(
    OUTPUT_FILES["pair_missing_case_ids"], index=False, encoding="utf-8-sig"
)

failed_coverage = pair_coverage_qc.loc[
    ~pair_coverage_qc["coverage_status"].eq("PASS")
].copy()
if len(failed_coverage):
    raise RuntimeError(
        "Unexpected primary-comparison coverage or identity loss. "
        f"Missing case IDs were exported to {OUTPUT_FILES['pair_missing_case_ids'].name}. "
        + failed_coverage.head(10).to_json(orient="records")
    )

if not pair_missing_case_ids.empty:
    raise RuntimeError("Missing case IDs were recorded despite passing coverage QC.")
if not paired_case_deltas_report_depth["method_family_match_pass"].all():
    raise RuntimeError("A comparison failed the pre-specified method-family matching rule.")

print("Pairing and coverage gate: PASS")


Pairing and coverage gate: PASS


In [10]:
# ==== Regime-Stratified Paired Bootstrap, Sign-Flip, McNemar, and Holm ====
def regime_stratified_paired_bootstrap_ci(scoped_frame, seed):
    required = {"delta", "regime"}
    missing = sorted(required.difference(scoped_frame.columns))
    if missing:
        raise RuntimeError(f"Stratified bootstrap missing columns: {missing}")
    if scoped_frame.empty:
        raise RuntimeError("Stratified bootstrap requires at least one paired case.")

    strata = []
    total_n = int(len(scoped_frame))
    for regime, group in scoped_frame.groupby("regime", sort=True, observed=True):
        values = pd.to_numeric(group["delta"], errors="raise").to_numpy(dtype=float)
        if values.size == 0 or not np.isfinite(values).all():
            raise RuntimeError(f"Invalid bootstrap stratum: {regime}")
        strata.append((str(regime), values, float(values.size / total_n)))

    rng = np.random.default_rng(seed)
    simulated_means = np.zeros(BOOTSTRAP_ITERATIONS, dtype=float)
    for start in range(0, BOOTSTRAP_ITERATIONS, SIMULATION_BATCH_SIZE):
        stop = min(start + SIMULATION_BATCH_SIZE, BOOTSTRAP_ITERATIONS)
        batch = np.zeros(stop - start, dtype=float)
        for _, values, weight in strata:
            indices = rng.integers(0, values.size, size=(stop - start, values.size))
            batch += weight * values[indices].mean(axis=1)
        simulated_means[start:stop] = batch

    lower, upper = np.quantile(simulated_means, [0.025, 0.975])
    weights = {regime: weight for regime, _, weight in strata}
    return float(lower), float(upper), weights


def regime_stratified_paired_sign_flip_p_value(scoped_frame, seed):
    required = {"delta", "regime"}
    missing = sorted(required.difference(scoped_frame.columns))
    if missing:
        raise RuntimeError(f"Stratified sign-flip test is missing columns: {missing}")
    if scoped_frame.empty:
        raise RuntimeError("Stratified sign-flip test requires paired cases.")

    strata = []
    total_n = int(len(scoped_frame))
    observed_mean = float(
        pd.to_numeric(scoped_frame["delta"], errors="raise").mean()
    )
    for regime, group in scoped_frame.groupby(
        "regime", sort=True, observed=True
    ):
        values = pd.to_numeric(
            group["delta"], errors="raise"
        ).to_numpy(dtype=float)
        if values.size == 0 or not np.isfinite(values).all():
            raise RuntimeError(f"Invalid sign-flip stratum: {regime}")
        strata.append((
            str(regime),
            values,
            float(values.size / total_n),
        ))

    rng = np.random.default_rng(int(seed))
    observed_statistic = abs(observed_mean)
    extreme_count = 0
    for start in range(0, SIGN_FLIP_ITERATIONS, SIMULATION_BATCH_SIZE):
        stop = min(start + SIMULATION_BATCH_SIZE, SIGN_FLIP_ITERATIONS)
        simulated_means = np.zeros(stop - start, dtype=float)
        for _, values, weight in strata:
            signs = rng.integers(
                0, 2, size=(stop - start, values.size), dtype=np.int8
            )
            signs = signs * 2 - 1
            simulated_means += weight * ((signs @ values) / values.size)
        extreme_count += int(np.count_nonzero(
            np.abs(simulated_means) >= observed_statistic - 1e-15
        ))

    p_value = (extreme_count + 1) / (SIGN_FLIP_ITERATIONS + 1)
    weights = {regime: weight for regime, _, weight in strata}
    return float(p_value), int(extreme_count), weights

def apply_principal_holm_adjustment(frame):
    adjusted = frame.copy()
    adjusted["holm_family_size"] = 0
    adjusted["holm_rank"] = 0
    adjusted["holm_adjusted_p_value"] = np.nan
    adjusted["holm_reject_0_05"] = False

    include = adjusted["confirmatory_holm_included"].astype(bool)
    principal = adjusted.loc[include].copy()
    observed_family = set(zip(
        principal["comparison_method_family"].astype(str),
        principal["contrast_code"].astype(str),
    ))
    if observed_family != EXPECTED_CONFIRMATORY_FAMILY:
        raise RuntimeError(
            "The confirmatory Holm family differs from the locked eight tests: "
            f"observed={sorted(observed_family)}, "
            f"expected={sorted(EXPECTED_CONFIRMATORY_FAMILY)}"
        )
    if len(principal) != 8 or principal["scope"].ne("overall").any():
        raise RuntimeError(
            "The confirmatory Holm family must contain exactly eight overall rows."
        )
    if principal["inference_role"].ne("principal_confirmatory").any():
        raise RuntimeError("A non-principal contrast entered the Holm family.")

    ordered = principal.sort_values(
        ["paired_sign_flip_permutation_p_value", "comparison_order"],
        kind="mergesort",
    )
    raw_values = ordered[
        "paired_sign_flip_permutation_p_value"
    ].to_numpy(dtype=float)
    family_size = int(len(ordered))
    scaled = (family_size - np.arange(family_size)) * raw_values
    holm_values = np.minimum(
        1.0, np.maximum.accumulate(scaled)
    )

    adjusted.loc[ordered.index, "holm_family_size"] = family_size
    adjusted.loc[ordered.index, "holm_rank"] = np.arange(
        1, family_size + 1
    )
    adjusted.loc[ordered.index, "holm_adjusted_p_value"] = holm_values
    adjusted.loc[ordered.index, "holm_reject_0_05"] = (
        holm_values <= HOLM_ALPHA
    )
    adjusted.loc[
        ordered.index, "multiplicity_family_id"
    ] = CONFIRMATORY_HOLM_FAMILY_ID
    adjusted.loc[
        ~include, "multiplicity_family_id"
    ] = adjusted.loc[~include, "inference_role"].astype(str)
    adjusted["holm_family_size"] = adjusted[
        "holm_family_size"
    ].astype(int)
    adjusted["holm_rank"] = adjusted["holm_rank"].astype(int)
    return adjusted



In [11]:
# ==== SELF-CHECK SC-3: Notebook 16 Locked Holm Family ====
def _sc3_make_locked_family_frame(extra_rows=None, drop_one=False):
    rows = []
    ordered_family = sorted(EXPECTED_CONFIRMATORY_FAMILY)
    p_values = [0.001, 0.004, 0.02, 0.03, 0.06, 0.08, 0.20, 0.50]
    if drop_one:
        ordered_family = ordered_family[:-1]
    for order, ((family, contrast), p_value) in enumerate(zip(ordered_family, p_values), start=1):
        rows.append({
            "comparison_method_family": family,
            "contrast_code": contrast,
            "scope": "overall",
            "inference_role": "principal_confirmatory",
            "confirmatory_holm_included": True,
            "paired_sign_flip_permutation_p_value": p_value,
            "comparison_order": order,
        })
    if extra_rows:
        rows.extend(extra_rows)
    return pd.DataFrame(rows)


def _sc3_expect_raises(label, fn):
    try:
        fn()
    except Exception:
        return {"test": label, "raised": True}
    raise RuntimeError(f"SC-3 negative test did not raise: {label}")


_sc3_adjusted = apply_principal_holm_adjustment(_sc3_make_locked_family_frame())
_sc3_holm = _sc3_adjusted.loc[_sc3_adjusted["confirmatory_holm_included"]].sort_values(
    ["holm_rank"], kind="mergesort"
)
if len(_sc3_holm) != 8:
    raise RuntimeError("SC-3 locked family membership count is not eight.")
if set(zip(_sc3_holm["comparison_method_family"], _sc3_holm["contrast_code"])) != EXPECTED_CONFIRMATORY_FAMILY:
    raise RuntimeError("SC-3 locked family membership mismatch.")
if _sc3_holm["holm_rank"].astype(int).tolist() != list(range(1, 9)):
    raise RuntimeError("SC-3 Holm ordering mismatch.")
if not np.diff(_sc3_holm["holm_adjusted_p_value"].to_numpy(dtype=float)).min(initial=0.0) >= -1e-15:
    raise RuntimeError("SC-3 Holm adjusted p-values are not monotone.")

_sc3_nb16_negative_results = [
    _sc3_expect_raises(
        "one missing family row",
        lambda: apply_principal_holm_adjustment(_sc3_make_locked_family_frame(drop_one=True)),
    ),
    _sc3_expect_raises(
        "one extra family row",
        lambda: apply_principal_holm_adjustment(_sc3_make_locked_family_frame(extra_rows=[{
            "comparison_method_family": "LightGBM",
            "contrast_code": "P2_Q_minus_P0",
            "scope": "overall",
            "inference_role": "principal_confirmatory",
            "confirmatory_holm_included": True,
            "paired_sign_flip_permutation_p_value": 0.9,
            "comparison_order": 99,
        }])),
    ),
]
SC3_NOTEBOOK16_SELF_CHECK_PASSED = True
sc3_notebook16_self_check_results = pd.DataFrame(_sc3_nb16_negative_results)
display(sc3_notebook16_self_check_results)
print("SC-3 Notebook 16 locked-family synthetic checks passed.")


,test,raised
0,one missing family row,True
1,one extra family row,True


SC-3 Notebook 16 locked-family synthetic checks passed.


In [12]:
# ==== Pre-Specified Paired Inference and Stage-1 Exact McNemar ====
inference_rows = []
coverage_lookup = pair_coverage_qc.set_index(["comparison_id", "scope"])

for comparison_spec in COMPARISON_SPECS:
    comparison_rows = paired_case_deltas_report_depth.loc[
        paired_case_deltas_report_depth["comparison_id"].eq(
            comparison_spec["comparison_id"]
        )
    ].copy()
    for scope in SCOPES:
        scoped = comparison_rows.loc[scope_mask(comparison_rows, scope)].copy()
        coverage = coverage_lookup.loc[(comparison_spec["comparison_id"], scope)]
        if len(scoped) != int(coverage["n_pairs"]):
            raise RuntimeError(
                f"Paired inference row count disagrees with coverage QC: "
                f"{comparison_spec['comparison_id']} / {scope}"
            )
        if scoped.empty:
            raise RuntimeError(
                f"No paired cases for pre-specified scope: "
                f"{comparison_spec['comparison_id']} / {scope}"
            )

        delta_values = scoped["delta"].to_numpy(dtype=float)
        bootstrap_seed = int(
            BOOTSTRAP_BASE_SEED
            + comparison_spec["comparison_order"] * 100
            + SCOPE_ORDER[scope]
        )
        sign_flip_seed = int(SIGN_FLIP_SEED)
        ci_lower, ci_upper, regime_weights = regime_stratified_paired_bootstrap_ci(
            scoped[["delta", "regime"]], bootstrap_seed
        )
        (
            sign_flip_p_value,
            extreme_count,
            sign_flip_regime_weights,
        ) = regime_stratified_paired_sign_flip_p_value(
            scoped[["delta", "regime"]], sign_flip_seed
        )
        positive_count = int(np.count_nonzero(delta_values > DELTA_TIE_ATOL))
        tie_count = int(np.count_nonzero(np.abs(delta_values) <= DELTA_TIE_ATOL))
        negative_count = int(np.count_nonzero(delta_values < -DELTA_TIE_ATOL))
        n_pairs = int(len(delta_values))
        confirmatory_holm_included = bool(
            scope == "overall"
            and comparison_spec["comparison_method_family"]
            in PRIMARY_METHOD_FAMILIES
            and comparison_spec["contrast_code"]
            in PRINCIPAL_CONTRAST_CODES
        )
        row_inference_role = comparison_spec["inference_role"]
        if (
            row_inference_role == "principal_confirmatory"
            and not confirmatory_holm_included
        ):
            row_inference_role = "principal_contrast_scope_unadjusted"

        inference_rows.append({
            "category_id": CATEGORY_ID,
            "category_label": CATEGORY_LABEL,
            "comparison_order": int(comparison_spec["comparison_order"]),
            "comparison_id": comparison_spec["comparison_id"],
            "contrast_code": comparison_spec["contrast_code"],
            "comparison_label": comparison_spec["comparison_label"],
            "inference_role": row_inference_role,
            "multiplicity_family_id": (
                CONFIRMATORY_HOLM_FAMILY_ID
                if confirmatory_holm_included
                else row_inference_role
            ),
            "confirmatory_holm_included": confirmatory_holm_included,
            "comparison_method_family": comparison_spec["comparison_method_family"],
            "method_family_label": comparison_spec["method_family_label"],
            "left_condition": comparison_spec["left_condition"],
            "right_condition": comparison_spec["right_condition"],
            "method_match_rule": comparison_spec["method_match_rule"],
            "method_family_match_pass": True,
            "scope": scope,
            "scope_order": int(SCOPE_ORDER[scope]),
            "metric_name": PRIMARY_METRIC_NAME,
            "metric_cutoff": PRIMARY_METRIC_CUTOFF,
            "candidate_pool_depth": REPORT_POOL_DEPTH,
            "n_left": int(coverage["n_left"]),
            "n_right": int(coverage["n_right"]),
            "n_pairs": n_pairs,
            "missing_left": int(coverage["missing_left"]),
            "missing_right": int(coverage["missing_right"]),
            "mean_left": float(scoped["left_metric_value"].mean()),
            "mean_right": float(scoped["right_metric_value"].mean()),
            "mean_delta": float(delta_values.mean()),
            "bootstrap_ci_95_lower": ci_lower,
            "bootstrap_ci_95_upper": ci_upper,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": bootstrap_seed,
            "bootstrap_unit": "paired_case_id_user_id_within_regime_strata",
            "bootstrap_regime_weights_json": json.dumps(
                regime_weights, ensure_ascii=False, sort_keys=True
            ),
            "paired_sign_flip_permutation_p_value": sign_flip_p_value,
            "sign_flip_extreme_count": extreme_count,
            "sign_flip_iterations": SIGN_FLIP_ITERATIONS,
            "sign_flip_seed": sign_flip_seed,
            "sign_flip_correction": "(extreme_count + 1) / (iterations + 1)",
            "sign_flip_unit": "paired_case_id_user_id_within_regime_strata",
            "sign_flip_regime_weights_json": json.dumps(
                sign_flip_regime_weights, ensure_ascii=False, sort_keys=True
            ),
            "positive_delta_count": positive_count,
            "positive_delta_rate": positive_count / n_pairs,
            "tie_count": tie_count,
            "tie_rate": tie_count / n_pairs,
            "negative_delta_count": negative_count,
            "negative_delta_rate": negative_count / n_pairs,
            "delta_tie_atol": DELTA_TIE_ATOL,
        })

paired_inference_all = apply_principal_holm_adjustment(pd.DataFrame(inference_rows))

INFERENCE_COLUMNS = [
    "category_id", "category_label", "comparison_order", "comparison_id",
    "contrast_code", "comparison_label", "inference_role",
    "multiplicity_family_id", "confirmatory_holm_included",
    "comparison_method_family", "method_family_label", "left_condition",
    "right_condition", "method_match_rule", "method_family_match_pass",
    "scope", "scope_order", "metric_name", "metric_cutoff",
    "candidate_pool_depth", "n_left", "n_right", "n_pairs", "missing_left",
    "missing_right", "mean_left", "mean_right", "mean_delta",
    "bootstrap_ci_95_lower", "bootstrap_ci_95_upper", "bootstrap_iterations",
    "bootstrap_seed", "bootstrap_unit", "bootstrap_regime_weights_json",
    "paired_sign_flip_permutation_p_value", "sign_flip_extreme_count",
    "sign_flip_iterations", "sign_flip_seed", "sign_flip_correction",
    "sign_flip_unit", "sign_flip_regime_weights_json",
    "holm_family_size", "holm_rank", "holm_adjusted_p_value",
    "holm_reject_0_05", "positive_delta_count", "positive_delta_rate",
    "tie_count", "tie_rate", "negative_delta_count", "negative_delta_rate",
    "delta_tie_atol",
]
paired_inference_all = paired_inference_all[INFERENCE_COLUMNS].sort_values(
    ["comparison_order", "scope_order"], kind="mergesort"
).reset_index(drop=True)
paired_inference_summary_report_depth = paired_inference_all.loc[
    paired_inference_all["scope"].eq("overall")
].reset_index(drop=True)
paired_inference_by_regime_report_depth = paired_inference_all.loc[
    ~paired_inference_all["scope"].eq("overall")
].reset_index(drop=True)

multiplicity_adjustment_summary = paired_inference_all.loc[
    paired_inference_all["confirmatory_holm_included"],
    [
        "category_id", "multiplicity_family_id", "holm_family_size",
        "holm_rank", "comparison_order", "comparison_id", "comparison_label",
        "comparison_method_family", "scope",
        "paired_sign_flip_permutation_p_value", "holm_adjusted_p_value",
        "holm_reject_0_05",
    ],
].copy()

# Stage 1 headline inference: exact McNemar on HitRate@1,000.
stage1_rows = pd.read_parquet(
    CANONICAL_METRICS_PATH,
    columns=CANONICAL_READ_COLUMNS,
).copy()
stage1_rows = stage1_rows.loc[
    stage1_rows["metric_name"].astype(str).eq(STAGE1_METRIC_NAME)
    & pd.to_numeric(stage1_rows["metric_cutoff"], errors="coerce").eq(STAGE1_METRIC_CUTOFF)
    & pd.to_numeric(stage1_rows["candidate_pool_depth"], errors="coerce").eq(STAGE1_POOL_DEPTH)
    & stage1_rows["stage_condition"].astype(str).isin(["P0", "P1-only"])
    & stage1_rows["method_family"].astype(str).eq("shared_retrieval_baseline")
].copy()
if stage1_rows.empty:
    raise RuntimeError("Stage-1 HitRate@1,000 rows are missing from Notebook 14 canonical metrics.")
if stage1_rows.duplicated(["case_id", "stage_condition"]).any():
    raise RuntimeError("Stage-1 McNemar rows are duplicated by case and condition.")

stage1_wide = stage1_rows.pivot(
    index="case_id", columns="stage_condition", values="metric_value"
)
if set(stage1_wide.columns) != {"P0", "P1-only"} or stage1_wide.isna().any().any():
    raise RuntimeError("Stage-1 McNemar pairing is incomplete.")
p0_success = stage1_wide["P0"].astype(float).gt(0.5)
p1_success = stage1_wide["P1-only"].astype(float).gt(0.5)
discordant_p1_only = int((p1_success & ~p0_success).sum())
discordant_p0_only = int((p0_success & ~p1_success).sum())
discordant_total = discordant_p1_only + discordant_p0_only
mcnemar_p = (
    float(binomtest(
        k=min(discordant_p1_only, discordant_p0_only),
        n=discordant_total,
        p=0.5,
        alternative="two-sided",
    ).pvalue)
    if discordant_total
    else 1.0
)
stage1_mcnemar_hitrate1000 = pd.DataFrame([{
    "category_id": CATEGORY_ID,
    "comparison_id": "P1_only_minus_P0_HitRate1000",
    "comparison_label": "P1-only − P0",
    "metric_name": STAGE1_METRIC_NAME,
    "metric_cutoff": STAGE1_METRIC_CUTOFF,
    "candidate_pool_depth": STAGE1_POOL_DEPTH,
    "n_pairs": int(len(stage1_wide)),
    "p1_only_success_p0_failure": discordant_p1_only,
    "p0_success_p1_only_failure": discordant_p0_only,
    "discordant_total": discordant_total,
    "mean_p0": float(p0_success.mean()),
    "mean_p1_only": float(p1_success.mean()),
    "hitrate_difference": float(p1_success.mean() - p0_success.mean()),
    "exact_mcnemar_p_value": mcnemar_p,
    "test_implementation": "exact_two_sided_binomial_on_discordant_pairs",
    "confirmatory_holm_included": False,
}])

print("Overall inference rows:", len(paired_inference_summary_report_depth))
print("Regime/non-cold inference rows:", len(paired_inference_by_regime_report_depth))
print("Stage-1 McNemar:", stage1_mcnemar_hitrate1000.to_dict("records")[0])


Overall inference rows: 17
Regime/non-cold inference rows: 68
Stage-1 McNemar: {'category_id': 'herbal', 'comparison_id': 'P1_only_minus_P0_HitRate1000', 'comparison_label': 'P1-only − P0', 'metric_name': 'HitRate', 'metric_cutoff': 1000, 'candidate_pool_depth': 1000, 'n_pairs': 1968, 'p1_only_success_p0_failure': 33, 'p0_success_p1_only_failure': 12, 'discordant_total': 45, 'mean_p0': 0.5609756097560976, 'mean_p1_only': 0.5716463414634146, 'hitrate_difference': 0.010670731707317027, 'exact_mcnemar_p_value': 0.002458900322096724, 'test_implementation': 'exact_two_sided_binomial_on_discordant_pairs', 'confirmatory_holm_included': False}


In [13]:
# ==== Final Validation, Exports, and Run Manifest ====
expected_inference_rows = len(COMPARISON_SPECS) * len(SCOPES)
if len(paired_inference_all) != expected_inference_rows:
    raise RuntimeError("Inference output does not contain every comparison × scope cell.")
if len(paired_inference_summary_report_depth) != len(COMPARISON_SPECS):
    raise RuntimeError("Overall inference output has an unexpected row count.")
if len(paired_inference_by_regime_report_depth) != (
    len(COMPARISON_SPECS) * (len(SCOPES) - 1)
):
    raise RuntimeError("By-regime inference output has an unexpected row count.")
if list(paired_inference_summary_report_depth.columns) != list(
    paired_inference_by_regime_report_depth.columns
):
    raise RuntimeError("Overall and by-regime inference schemas differ.")
if not paired_inference_all["method_family_match_pass"].all():
    raise RuntimeError("Method-family matching failed.")
if not paired_inference_all["candidate_pool_depth"].eq(REPORT_POOL_DEPTH).all():
    raise RuntimeError("An inferential row uses a non-primary pool depth.")
if not paired_inference_all["metric_name"].eq(PRIMARY_METRIC_NAME).all():
    raise RuntimeError("An inferential row uses a non-primary metric.")
if not paired_inference_all["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF).all():
    raise RuntimeError("An inferential row uses a non-primary metric cutoff.")
if not np.isclose(
    paired_inference_all["mean_left"] - paired_inference_all["mean_right"],
    paired_inference_all["mean_delta"],
    rtol=0.0,
    atol=1e-12,
).all():
    raise RuntimeError("Mean delta does not equal paired mean_left − mean_right.")
direction_total = (
    paired_inference_all["positive_delta_count"]
    + paired_inference_all["tie_count"]
    + paired_inference_all["negative_delta_count"]
)
if not direction_total.eq(paired_inference_all["n_pairs"]).all():
    raise RuntimeError("Positive/tie/negative counts do not sum to n_pairs.")
direction_rate_total = (
    paired_inference_all["positive_delta_rate"]
    + paired_inference_all["tie_rate"]
    + paired_inference_all["negative_delta_rate"]
)
if not np.isclose(direction_rate_total, 1.0, rtol=0.0, atol=1e-12).all():
    raise RuntimeError("Positive/tie/negative rates do not sum to one.")
if not paired_inference_all["paired_sign_flip_permutation_p_value"].between(
    1.0 / (SIGN_FLIP_ITERATIONS + 1), 1.0, inclusive="both"
).all():
    raise RuntimeError("A finite-corrected sign-flip p-value is outside its valid range.")
_holm_rows = paired_inference_all.loc[paired_inference_all["confirmatory_holm_included"]]
if not _holm_rows["holm_adjusted_p_value"].between(0.0, 1.0, inclusive="both").all():
    raise RuntimeError("A confirmatory Holm-adjusted p-value is outside [0, 1].")
if not _holm_rows["holm_adjusted_p_value"].ge(
    _holm_rows["paired_sign_flip_permutation_p_value"] - 1e-15
).all():
    raise RuntimeError("A confirmatory Holm-adjusted p-value is smaller than its raw p-value.")

confirmatory_holm_rows = paired_inference_all.loc[
    paired_inference_all["confirmatory_holm_included"]
]
if len(confirmatory_holm_rows) != 8:
    raise RuntimeError("Confirmatory Holm family does not contain exactly eight overall Stage-2 tests.")
if confirmatory_holm_rows["holm_family_size"].ne(8).any():
    raise RuntimeError("Confirmatory Holm family size is not eight.")
if confirmatory_holm_rows["holm_adjusted_p_value"].isna().any():
    raise RuntimeError("A confirmatory Holm-adjusted p-value is missing.")
observed_confirmatory_family = set(zip(
    confirmatory_holm_rows["comparison_method_family"].astype(str),
    confirmatory_holm_rows["contrast_code"].astype(str),
))
if observed_confirmatory_family != EXPECTED_CONFIRMATORY_FAMILY:
    raise RuntimeError("The final Holm rows do not match the locked family.")

_holm_ordered = confirmatory_holm_rows.sort_values(
    ["paired_sign_flip_permutation_p_value", "comparison_order"],
    kind="mergesort",
)
if _holm_ordered["holm_rank"].astype(int).tolist() != list(range(1, 9)):
    raise RuntimeError("Holm ranks are not the correct ordered sequence 1..8.")
_holm_raw = _holm_ordered[
    "paired_sign_flip_permutation_p_value"
].to_numpy(dtype=float)
_holm_expected = np.minimum(
    1.0,
    np.maximum.accumulate((8 - np.arange(8)) * _holm_raw),
)
if not np.allclose(
    _holm_ordered["holm_adjusted_p_value"].to_numpy(dtype=float),
    _holm_expected,
    rtol=0.0,
    atol=1e-15,
):
    raise RuntimeError("Holm adjusted p-values do not follow the locked order.")
if np.any(np.diff(
    _holm_ordered["holm_adjusted_p_value"].to_numpy(dtype=float)
) < -1e-15):
    raise RuntimeError("Holm adjusted p-values are not monotone by Holm rank.")
exploratory_rows = paired_inference_all.loc[
    ~paired_inference_all["confirmatory_holm_included"]
]
if exploratory_rows["holm_adjusted_p_value"].notna().any():
    raise RuntimeError("Exploratory rows received confirmatory Holm adjustment.")
if not paired_inference_all["sign_flip_iterations"].eq(
    SIGN_FLIP_ITERATIONS
).all():
    raise RuntimeError("A paired test used the wrong sign-flip count.")
if not paired_inference_all["sign_flip_seed"].eq(SIGN_FLIP_SEED).all():
    raise RuntimeError("A paired test used a seed other than 42.")
expected_holm_family_sizes = {CONFIRMATORY_HOLM_FAMILY_ID: 8}

paired_case_deltas_report_depth.to_parquet(
    OUTPUT_FILES["paired_case_deltas_report_depth"], index=False
)
paired_inference_summary_report_depth.to_csv(
    OUTPUT_FILES["paired_inference_summary_report_depth"],
    index=False,
    encoding="utf-8-sig",
)
paired_inference_by_regime_report_depth.to_csv(
    OUTPUT_FILES["paired_inference_by_regime_report_depth"],
    index=False,
    encoding="utf-8-sig",
)
pair_coverage_qc.to_csv(
    OUTPUT_FILES["pair_coverage_qc"], index=False, encoding="utf-8-sig"
)
pair_missing_case_ids.to_csv(
    OUTPUT_FILES["pair_missing_case_ids"], index=False, encoding="utf-8-sig"
)
multiplicity_adjustment_summary.to_csv(
    OUTPUT_FILES["multiplicity_adjustment_summary"],
    index=False,
    encoding="utf-8-sig",
)
stage1_mcnemar_hitrate1000.to_csv(
    OUTPUT_FILES["stage1_mcnemar_hitrate1000"],
    index=False,
    encoding="utf-8-sig",
)

validation_results = {
    "notebook14_manifest_required_checks_passed": True,
    "notebook14_hashes_verified": True,
    "notebook14_ready_for_downstream": True,
    "exact_locked_holm_family_verified": True,
    "holm_order_verified": True,
    "notebook14_canonical_rank_qc_passed": True,
    "notebook14_paired_universe_qc_passed": True,
    "authoritative_inputs_only": True,
    "metric_fixed_before_outcome_load": True,
    "pool_depth_fixed_before_outcome_load": True,
    "one_row_per_case_condition_method_family": True,
    "one_case_per_user": True,
    "target_absent_ndcg_zero_preserved": True,
    "strict_case_id_pairing": True,
    "regime_consistency_across_pairs": True,
    "no_unmatched_primary_cases": True,
    "all_prior_comparisons_method_family_matched": True,
    "no_method_or_winner_selection": True,
    "no_rank_reconstruction": True,
    "finite_simulation_correction_applied": True,
    "holm_adjustment_applied_to_locked_eight_principal_tests": True,
    "stage1_exact_mcnemar_hitrate1000_completed": True,
    "regime_stratified_bootstrap_completed": True,
    "seed_root_42_used": True,
    "overall_and_strong_scopes_reported": True,
    "output_columns_unique": True,
}

run_manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "latest_revision": f"locked four-family principal Holm inference ({REVISION_DATE})",
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "analysis_scope": "statistical_inference_only",
    "authoritative_source_notebook": "14_pipeline_aggregate",
    "input_paths": {
        "canonical_per_case_metrics": str(CANONICAL_METRICS_PATH),
        "canonical_primary_method_grid": str(CANONICAL_PRIMARY_PATH),
        "pipeline_manifest": str(PIPELINE_MANIFEST_PATH),
    },
    "input_sha256": {
        "canonical_per_case_metrics": file_sha256(CANONICAL_METRICS_PATH),
        "canonical_primary_method_grid": file_sha256(CANONICAL_PRIMARY_PATH),
        "pipeline_manifest": file_sha256(PIPELINE_MANIFEST_PATH),
    },
    "outcome_row_filter_fixed_before_load": {
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "candidate_pool_depth": REPORT_POOL_DEPTH,
    },
    "comparison_specs": COMPARISON_SPECS,
    "scopes": SCOPES,
    "non_cold_definition": "regime != cold",
    "bootstrap": {
        "unit": "case_id resampled within regime strata and recombined at empirical analysis-population weights",
        "iterations": BOOTSTRAP_ITERATIONS,
        "base_seed": BOOTSTRAP_BASE_SEED,
        "confidence_interval": "percentile_2.5_97.5",
    },
    "sign_flip_test": {
        "statistic": "absolute mean paired delta",
        "two_sided": True,
        "iterations": SIGN_FLIP_ITERATIONS,
        "seed": SIGN_FLIP_SEED,
        "finite_simulation_correction": "(extreme_count + 1) / (iterations + 1)",
    },
    "multiplicity": {
        "method": "Holm",
        "alpha": HOLM_ALPHA,
        "family_definition": (
            "{P2-P minus P2-Q, Full minus P2-P} × "
            "{Shannon, GAM, LightGBM, Transformer} = 8 overall tests per category"
        ),
        "locked_family_members": [
            {
                "method_family": family,
                "contrast_code": contrast_code,
            }
            for family, contrast_code in sorted(EXPECTED_CONFIRMATORY_FAMILY)
        ],
        "P2_Q_minus_P0_outside_holm": True,
        "Full_minus_P2_Q_outside_holm_exploratory": True,
        "family_sizes": expected_holm_family_sizes,
        "stage1_mcnemar_excluded": True,
        "regime_and_non_cold_rows_exploratory_unadjusted": True,
    },
    "stage1_exact_mcnemar": {
        "metric_name": STAGE1_METRIC_NAME,
        "metric_cutoff": STAGE1_METRIC_CUTOFF,
        "candidate_pool_depth": STAGE1_POOL_DEPTH,
        "comparison": "P1-only minus P0",
        "output_path": str(OUTPUT_FILES["stage1_mcnemar_hitrate1000"]),
    },
    "tie_definition": f"abs(delta) <= {DELTA_TIE_ATOL}",
    "notebook14_stage1_winner_lineage": stage1_winner_lineage,
    "category_specific_method_lineage": method_lineage,
    "case_count": int(primary_metrics["case_id"].nunique()),
    "case_counts_by_regime": {
        str(key): int(value)
        for key, value in (
            primary_metrics[["case_id", "regime"]]
            .drop_duplicates("case_id")["regime"]
            .value_counts()
            .sort_index()
            .to_dict()
        ).items()
    },
    "output_paths": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "output_schemas": {
        "paired_case_deltas_report_depth": list(paired_case_deltas_report_depth.columns),
        "paired_inference_summary_report_depth": list(paired_inference_summary_report_depth.columns),
        "paired_inference_by_regime_report_depth": list(paired_inference_by_regime_report_depth.columns),
        "pair_coverage_qc": list(pair_coverage_qc.columns),
        "pair_missing_case_ids": list(pair_missing_case_ids.columns),
        "multiplicity_adjustment_summary": list(multiplicity_adjustment_summary.columns),
        "stage1_mcnemar_hitrate1000": list(stage1_mcnemar_hitrate1000.columns),
    },
    "validation_results": validation_results,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(OUTPUT_FILES["run_manifest"], run_manifest)

missing_outputs = [str(path) for path in OUTPUT_FILES.values() if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Required paired-inference outputs were not written: {missing_outputs}")

print("Paired inference outputs written to:", OUT_DIR)
print("Validation: PASS")


Paired inference outputs written to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/paired_significance_summary
Validation: PASS
